# 🧪 00 — Setup & Evidence

Build a tiny disposable Delta table plus a **separate evidence ledger**. The ledger stores versions, commit timestamps, checksums, schema fingerprints, and physical-file signatures so later labs can abuse the table without erasing the evidence.

> Disposable lab only. The Morlocks get less careful from here.

In [ ]:
# --- Lab configuration -------------------------------------------------------
# Explicitly choose a disposable schema and inspect the target before cleanup.
CATALOG = spark.catalog.currentCatalog()
SCHEMA = "morlocks_time_travel_lab"
EXPERIMENT_ID = "morlock-time-travel-001"
DATA_TABLE = "morlock_customer_state"
EVIDENCE_TABLE = "morlock_time_travel_evidence"

spark.conf.set("spark.sql.session.timeZone", "UTC")

def qi(x: str) -> str:
    return f"`{x.replace('`', '``')}`"

def fqn(catalog: str, schema: str, table: str) -> str:
    parts = (schema, table) if catalog == "spark_catalog" else (catalog, schema, table)
    return ".".join(qi(x) for x in parts)

DATA_FQN = fqn(CATALOG, SCHEMA, DATA_TABLE)
EVIDENCE_FQN = fqn(CATALOG, SCHEMA, EVIDENCE_TABLE)

print("Data table    :", DATA_FQN)
print("Evidence table:", EVIDENCE_FQN)

## 1. Create the disposable lab schema, then reset its two named tables

In [ ]:
schema_name = qi(SCHEMA) if CATALOG == "spark_catalog" else f"{qi(CATALOG)}.{qi(SCHEMA)}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
print("Disposable schema selected:", f"{CATALOG}.{SCHEMA}")
spark.sql(f"DROP TABLE IF EXISTS {DATA_FQN}")
spark.sql(f"DROP TABLE IF EXISTS {EVIDENCE_FQN}")

## 2. Helpers + format-neutral evidence schema

In [ ]:
from pyspark.sql import functions as F, types as T
from datetime import datetime, timezone
import hashlib, json

BUSINESS_COLUMNS = [
    "customer_id", "name", "segment", "balance",
    "status", "event_time", "source_system"
]

EVIDENCE_SCHEMA = T.StructType([
    T.StructField("experiment_id", T.StringType(), False),
    T.StructField("operation_id", T.LongType(), False),
    T.StructField("operation_name", T.StringType(), False),
    T.StructField("wall_clock_timestamp", T.TimestampType(), False),
    T.StructField("event_time_min", T.TimestampType(), True),
    T.StructField("event_time_max", T.TimestampType(), True),
    T.StructField("delta_version", T.LongType(), True),
    T.StructField("delta_commit_timestamp", T.TimestampType(), True),
    T.StructField("delta_operation", T.StringType(), True),
    T.StructField("converted_delta_version", T.LongType(), True),
    T.StructField("converted_delta_timestamp", T.TimestampType(), True),
    T.StructField("iceberg_snapshot_id", T.StringType(), True),
    T.StructField("iceberg_metadata_path", T.StringType(), True),
    T.StructField("hudi_instant_or_identifier", T.StringType(), True),
    T.StructField("row_count", T.LongType(), True),
    T.StructField("logical_checksum", T.StringType(), True),
    T.StructField("schema_json", T.StringType(), True),
    T.StructField("schema_fingerprint", T.StringType(), True),
    T.StructField("active_file_count", T.LongType(), True),
    T.StructField("active_file_paths_hash", T.StringType(), True),
    T.StructField("runtime_version", T.StringType(), True),
    T.StructField("spark_version", T.StringType(), True),
    T.StructField("table_properties_json", T.StringType(), True),
    T.StructField("notes", T.StringType(), True),
])

def logical_checksum(df, columns=BUSINESS_COLUMNS):
    """Deterministic checksum for this intentionally tiny lab table."""
    row_json = F.to_json(
        F.struct(*[F.col(c) for c in columns]),
        options={"ignoreNullFields": "false"},
    )
    return (
        df.select(F.sha2(row_json, 256).alias("_h"))
          .agg(F.sha2(F.concat_ws("||", F.sort_array(F.collect_list("_h"))), 256).alias("checksum"))
          .first()["checksum"]
    )

def schema_signature(df):
    raw = df.schema.json()
    return raw, hashlib.sha256(raw.encode()).hexdigest()

def observed_active_files(df):
    paths = sorted(
        r["path"] for r in
        df.select(F.input_file_name().alias("path"))
          .where(F.col("path") != "")
          .distinct().collect()
    )
    return len(paths), hashlib.sha256("\n".join(paths).encode()).hexdigest()

def table_properties_json(table_fqn):
    props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {table_fqn}").collect()}
    return json.dumps(props, sort_keys=True)

def latest_history(table_fqn):
    return spark.sql(f"DESCRIBE HISTORY {table_fqn} LIMIT 1").first()

def history_for_version(table_fqn, version):
    rows = spark.sql(f"DESCRIBE HISTORY {table_fqn}").where(F.col("version") == int(version)).collect()
    if not rows:
        raise ValueError(f"Version {version} not found")
    return rows[0]

def runtime_version():
    try:
        return spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion")
    except Exception:
        return "unknown"

def next_operation_id():
    row = spark.table(EVIDENCE_FQN).agg(F.max("operation_id").alias("m")).first()
    return 1 if row["m"] is None else int(row["m"]) + 1

def record_evidence(operation_name, notes=""):
    df = spark.table(DATA_FQN)
    hist = latest_history(DATA_FQN)
    bounds = df.agg(F.min("event_time").alias("lo"), F.max("event_time").alias("hi")).first()
    schema_json, schema_hash = schema_signature(df)
    file_count, file_hash = observed_active_files(df)

    row = {
        "experiment_id": EXPERIMENT_ID,
        "operation_id": next_operation_id(),
        "operation_name": operation_name,
        "wall_clock_timestamp": datetime.now(timezone.utc).replace(tzinfo=None),
        "event_time_min": bounds["lo"],
        "event_time_max": bounds["hi"],
        "delta_version": int(hist["version"]),
        "delta_commit_timestamp": hist["timestamp"],
        "delta_operation": hist["operation"],
        "converted_delta_version": None,
        "converted_delta_timestamp": None,
        "iceberg_snapshot_id": None,
        "iceberg_metadata_path": None,
        "hudi_instant_or_identifier": None,
        "row_count": df.count(),
        "logical_checksum": logical_checksum(df),
        "schema_json": schema_json,
        "schema_fingerprint": schema_hash,
        "active_file_count": file_count,
        "active_file_paths_hash": file_hash,
        "runtime_version": runtime_version(),
        "spark_version": spark.version,
        "table_properties_json": table_properties_json(DATA_FQN),
        "notes": notes,
    }
    out = spark.createDataFrame([row], EVIDENCE_SCHEMA)
    out.write.format("delta").mode("append").saveAsTable(EVIDENCE_FQN)
    return out

## 3. Create the black box first

In [ ]:
spark.createDataFrame([], EVIDENCE_SCHEMA) \
     .write.format("delta").mode("append").saveAsTable(EVIDENCE_FQN)

## 4. Create the initial Delta state

In [ ]:
initial_rows = [
    (1, "Alice Eloi",    "gold",   1250.00, "ACTIVE", "2026-08-12T08:15:00Z", "crm"),
    (2, "Bob Wells",     "silver",  275.50, "ACTIVE", "2026-08-12T08:17:00Z", "crm"),
    (3, "Clara Morlock", "bronze",   42.00, "ACTIVE", "2026-08-12T08:20:00Z", "billing"),
    (4, "David George",  "gold",   8900.00, "ACTIVE", "2026-08-12T08:24:00Z", "billing"),
    (5, "Eve Traveller", "silver",  640.25, "PAUSED", "2026-08-12T08:31:00Z", "crm"),
    (6, "Frank Chronos", "gold",   3100.00, "ACTIVE", "2026-08-12T08:44:00Z", "crm"),
]
schema = "customer_id LONG, name STRING, segment STRING, balance DOUBLE, status STRING, event_time STRING, source_system STRING"
df = spark.createDataFrame(initial_rows, schema=schema).withColumn("event_time", F.to_timestamp("event_time"))
df.write.format("delta").mode("append").saveAsTable(DATA_FQN)

## 5. Record the baseline evidence

In [ ]:
baseline_evidence = record_evidence(
    "initial_load",
    "Baseline state before the deliberately late record arrives."
)
display(baseline_evidence)

## 6. Inspect and sanity-check the baseline

In [ ]:
baseline = (spark.table(EVIDENCE_FQN)
    .where(F.col("operation_name") == "initial_load")
    .orderBy(F.col("operation_id").desc()).first())

BASELINE_VERSION = int(baseline["delta_version"])

display(spark.table(DATA_FQN).orderBy("customer_id"))
display(spark.sql(f"DESCRIBE HISTORY {DATA_FQN}"))

assert baseline["row_count"] == 6
assert spark.sql(f"SELECT count(*) n FROM {DATA_FQN} VERSION AS OF {BASELINE_VERSION}").first()["n"] == 6
assert baseline["logical_checksum"]
assert baseline["schema_fingerprint"]

print("✅ Baseline Delta version:", BASELINE_VERSION)
print("✅ Logical checksum      :", baseline["logical_checksum"])
print("✅ Evidence table        :", EVIDENCE_FQN)

## Result

Leave both tables in place. `01_three_clocks.ipynb` will append one record whose business event happened **before** the baseline, then prove which later Delta version first contains it.